# RAG-ассистент Фильмоскоп

Минимальный проектный notebook. Текущая реализация закрывает шаг 3 плана: два способа чанкинга, эмбеддинги, FAISS на CPU по умолчанию, retrieval-метрики и выбор лучшей стратегии. GPU FAISS используется только опционально, если он доступен в окружении.


## Константы и импорты


In [1]:
import json
import re
import random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ROOT = Path.cwd()
DATA_PATH = ROOT / "filmoskop_kb.json"
EMBED_MODEL = "intfloat/multilingual-e5-base"
RECALL_KS = [1, 3, 5, 10, 20]
TOP_K = 20

## Проверка зависимостей для шага 3


In [2]:
try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        "Не установлена зависимость: langchain-text-splitters. "
        "Установите её в том же окружении/Jupyter kernel: "
        "python -m pip install langchain-text-splitters"
    ) from e

from sentence_transformers import SentenceTransformer
import faiss
import torch

print("Версия FAISS:", getattr(faiss, "__version__", "unknown"))

FAISS_GPU_AVAILABLE = (
    hasattr(faiss, "StandardGpuResources")
    and hasattr(faiss, "get_num_gpus")
    and faiss.get_num_gpus() > 0
)

if FAISS_GPU_AVAILABLE:
    print("Доступно GPU для FAISS:", faiss.get_num_gpus())
else:
    print("GPU FAISS недоступен; используется CPU FAISS.")

TORCH_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Устройство PyTorch для эмбеддингов:", TORCH_DEVICE)


Версия FAISS: 1.16.0
GPU FAISS недоступен; используется CPU FAISS.
Устройство PyTorch для эмбеддингов: cuda


## Диагностика датасета и базовая очистка для шага 3

Эта ячейка загружает базу знаний для локального тестового прогона. Простая очистка и сводка по данным нужны перед чанкингом и построением индекса поиска.


In [3]:
with DATA_PATH.open(encoding="utf-8") as f:
    data = json.load(f)

docs = data["docs"]
queries = data["queries"]

print("Документы:", len(docs), "| Вопросы:", len(queries))
print("Форматы:", dict(Counter(d["format"] for d in docs)))
print("Категории:", dict(Counter(d["category"] for d in docs)))
print("Типы вопросов:", dict(Counter(q["kind"] for q in queries)))


Документы: 181 | Вопросы: 66
Форматы: {'text': 132, 'markdown': 28, 'html': 21}
Категории: {'Подписка и оплата': 38, 'Функции и контент': 34, 'Аккаунт и профили': 35, 'Устройства и качество': 42, 'Ошибки воспроизведения': 32}
Типы вопросов: {'exact': 15, 'semantic': 25, 'multi': 16, 'gap': 10}


In [4]:
def clean_text(raw: str) -> str:
    """Очищает исходный документ и нормализует пробелы.

    Args:
        raw: Исходный текст документа из базы знаний.

    Returns:
        Очищенный текст без простых HTML/Markdown-разметок и лишних пробелов.
    """
    text = raw or ""
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"!\[[^\]]*\]\([^)]*\)", " ", text)
    text = re.sub(r"\[([^\]]+)\]\([^)]*\)", r"\1", text)
    text = re.sub(r"[`*_#>\-]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

clean_docs = []
for d in docs:
    clean_docs.append({
        "doc_id": d["doc_id"],
        "title": d["title"],
        "category": d["category"],
        "text": clean_text(d["raw"]),
    })

pd.DataFrame({
    "metric": ["docs", "queries", "median_clean_len"],
    "value": [len(clean_docs), len(queries), int(np.median([len(d["text"]) for d in clean_docs]))],
})


,metric,value
0,docs,181
1,queries,66
2,median_clean_len,662


## Шаг 3. Два варианта чанкинга


In [5]:
CHUNK_STRATEGIES = {
    "small_500_100": {"chunk_size": 500, "chunk_overlap": 100},
    "medium_900_150": {"chunk_size": 900, "chunk_overlap": 150},
}

def make_chunks(clean_docs, chunk_size: int, chunk_overlap: int):
    """Делит тексты на чанки с метаданными документов.

    Args:
        clean_docs: Список очищенных документов с ключами `doc_id`, `title`,
            `category` и `text`.
        chunk_size: Максимальный размер чанка в символах.
        chunk_overlap: Количество символов перекрытия между чанками.

    Returns:
        Список словарей с текстом чанка и метаданными `doc_id`, `title`,
        `category`, `chunk_id`.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = []
    for doc in clean_docs:
        parts = splitter.split_text(f"{doc['title']}\n{doc['text']}")
        for i, part in enumerate(parts):
            chunks.append({
                "page_content": part,
                "doc_id": doc["doc_id"],
                "title": doc["title"],
                "category": doc["category"],
                "chunk_id": i,
            })
    return chunks

chunk_sets = {}
for name, params in CHUNK_STRATEGIES.items():
    chunk_sets[name] = make_chunks(clean_docs, **params)
    lengths = [len(c["page_content"]) for c in chunk_sets[name]]
    print(name, "чанков:", len(lengths), "медианная длина:", int(np.median(lengths)))


small_500_100 чанков: 502 медианная длина: 374
medium_900_150 чанков: 323 медианная длина: 462


## Шаг 3. Эмбеддинги и FAISS-индексы


In [6]:
embedder = SentenceTransformer(EMBED_MODEL, device=TORCH_DEVICE)

def encode_passages(texts):
    """Создаёт векторы документов для модели E5.

    Args:
        texts: Список текстов чанков.

    Returns:
        Массив эмбеддингов `float32`, нормализованных для поиска по
        скалярному произведению в FAISS.
    """
    vecs = embedder.encode(
        ["passage: " + t for t in texts],
        normalize_embeddings=True,
        batch_size=32,
        show_progress_bar=True,
    )
    return np.asarray(vecs, dtype="float32")

def encode_queries(texts):
    """Создаёт нормализованные векторы запросов для модели E5.

    Args:
        texts: Список текстов запросов.

    Returns:
        Массив эмбеддингов `float32`, нормализованных для поиска по
        скалярному произведению в FAISS.
    """
    vecs = embedder.encode(
        ["query: " + t for t in texts],
        normalize_embeddings=True,
        batch_size=32,
        show_progress_bar=False,
    )
    return np.asarray(vecs, dtype="float32")

def build_faiss_index(vectors):
    """Создаёт индекс FAISS `IndexFlatIP` для нормализованных эмбеддингов.

    Args:
        vectors: Матрица dense-эмбеддингов `float32` формы
            `(количество_чанков, размерность_эмбеддинга)`.

    Returns:
        Индекс FAISS с добавленными векторами. По умолчанию используется CPU
        FAISS; GPU FAISS используется только если он доступен в окружении.
    """
    cpu_index = faiss.IndexFlatIP(vectors.shape[1])
    if FAISS_GPU_AVAILABLE:
        res = faiss.StandardGpuResources()
        index = faiss.index_cpu_to_gpu(res, 0, cpu_index)
        index.add(vectors)
        return index

    cpu_index.add(vectors)
    return cpu_index

indexes = {}
embeddings = {}
for name, chunks in chunk_sets.items():
    embeddings[name] = encode_passages([c["page_content"] for c in chunks])
    indexes[name] = build_faiss_index(embeddings[name])
    print(name, embeddings[name].shape, "размер индекса", indexes[name].ntotal)


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

small_500_100 (502, 768) размер индекса 502


Batches:   0%|          | 0/11 [00:00<?, ?it/s]

medium_900_150 (323, 768) размер индекса 323


## Шаг 3. Retrieval-метрики


In [7]:
eval_queries = [q for q in queries if q["kind"] != "gap"]
query_vecs = encode_queries([q["query"] for q in eval_queries])

def retrieved_doc_ids(strategy: str, q_vec, k: int):
    """Возвращает уникальные `doc_id` из top-k найденных чанков.

    Args:
        strategy: Название стратегии чанкинга из `chunk_sets` и `indexes`.
        q_vec: Вектор запроса.
        k: Количество чанков, запрашиваемое у FAISS.

    Returns:
        Список уникальных идентификаторов документов в порядке релевантности.
    """
    _, idx = indexes[strategy].search(q_vec.reshape(1, -1), k)
    doc_ids = []
    seen = set()
    for i in idx[0]:
        doc_id = chunk_sets[strategy][int(i)]["doc_id"]
        if doc_id not in seen:
            doc_ids.append(doc_id)
            seen.add(doc_id)
    return doc_ids

def precision_at_k(found, relevant):
    """Считает precision для найденных `doc_id`.

    Args:
        found: Список найденных идентификаторов документов.
        relevant: Список эталонных идентификаторов документов.

    Returns:
        Доля найденных документов, которые входят в список релевантных.
    """
    if not found:
        return 0.0
    rel = set(relevant)
    return sum(doc_id in rel for doc_id in found) / len(found)

def recall_at_k(found, relevant):
    """Считает recall для найденных `doc_id`.

    Args:
        found: Список найденных идентификаторов документов.
        relevant: Список эталонных идентификаторов документов.

    Returns:
        Доля релевантных документов, найденных поиском.
    """
    rel = set(relevant)
    if not rel:
        return 0.0
    return len(set(found) & rel) / len(rel)

def mrr(found, relevant):
    """Считает reciprocal rank первого релевантного документа.

    Args:
        found: Список найденных идентификаторов документов в порядке релевантности.
        relevant: Список эталонных идентификаторов документов.

    Returns:
        Значение reciprocal rank или `0.0`, если релевантных документов нет в выдаче.
    """
    rel = set(relevant)
    for rank, doc_id in enumerate(found, start=1):
        if doc_id in rel:
            return 1 / rank
    return 0.0

def evaluate_strategy(strategy: str):
    """Считает retrieval-метрики для одной стратегии чанкинга.

    Args:
        strategy: Название стратегии чанкинга из `chunk_sets` и `indexes`.

    Returns:
        `pandas.DataFrame` по вопросам со списками найденных и релевантных документов.
    """
    rows = []
    for q, q_vec in zip(eval_queries, query_vecs):
        found20 = retrieved_doc_ids(strategy, q_vec, TOP_K)
        row = {
            "strategy": strategy,
            "kind": q["kind"],
            "query": q["query"],
            "relevant_docs": q["relevant_docs"],
            "found_doc_ids": found20,
            "precision@5": precision_at_k(found20[:5], q["relevant_docs"]),
            "recall@5": recall_at_k(found20[:5], q["relevant_docs"]),
            "MRR": mrr(found20, q["relevant_docs"]),
        }
        for k in RECALL_KS:
            row[f"recall@{k}"] = recall_at_k(found20[:k], q["relevant_docs"])
        rows.append(row)
    return pd.DataFrame(rows)

metrics_by_query = pd.concat([evaluate_strategy(name) for name in chunk_sets], ignore_index=True)
summary = metrics_by_query.groupby("strategy")[["precision@5", "recall@5", "MRR", "recall@1", "recall@3", "recall@10", "recall@20"]].mean().reset_index()
summary


,strategy,precision@5,recall@5,MRR,recall@1,recall@3,recall@10,recall@20
0,medium_900_150,0.146429,0.562500,0.417792,0.169643,0.5,0.669643,0.785714
1,small_500_100,0.153571,0.589286,0.465340,0.258929,0.5,0.687500,0.776786


In [8]:
by_kind = metrics_by_query.groupby(["strategy", "kind"])[["precision@5", "recall@5", "MRR"]].mean().reset_index()
by_kind

,strategy,kind,precision@5,recall@5,MRR
0,medium_900_150,exact,0.173333,0.833333,0.497222
1,medium_900_150,multi,0.200000,0.500000,0.472070
2,medium_900_150,semantic,0.096000,0.440000,0.335397
3,small_500_100,exact,0.173333,0.833333,0.638889
4,small_500_100,multi,0.212500,0.531250,0.520908
5,small_500_100,semantic,0.104000,0.480000,0.325648


## Шаг 3. Анализ примеров чанков и выбор стратегии


In [9]:
for name, chunks in chunk_sets.items():
    print("\n===", name, "===")
    for c in chunks[:3]:
        print(f"doc_id={c['doc_id']} chunk_id={c['chunk_id']} title={c['title']}")
        print(c["page_content"][:450].replace("\n", " "))
        print("---")


=== small_500_100 ===
doc_id=0 chunk_id=0 title=Промокоды
Промокоды Промокод даёт скидку на подписку или бесплатные дни. Ввести его можно в разделе «Подписка» — поле «У меня есть промокод» — или при оформлении подписки. Правила простые: один промокод на один аккаунт, скидки по промокодам не суммируются между собой и с другими акциями, у каждого кода есть срок действия — он указан в условиях акции. Если код не применяется, проверьте срок действия и то, что он ещё не был использован на этом аккаунте.
---
doc_id=1 chunk_id=0 title=Синхронизация между устройствами
Синхронизация между устройствами
---
doc_id=1 chunk_id=1 title=Синхронизация между устройствами
Всё, что вы делаете в «Фильмоскопе», хранится не на устройстве, а в аккаунте. Поэтому телефон, планшет, браузер и телевизор показывают одно и то же: вошли — и сервис выглядит привычно, даже если устройство чужое. Общее для аккаунта: подписка и тариф, способ оплаты, список привязанных устройств. Это не зависит ни от профиля, ни от

In [10]:
best_strategy = summary.sort_values(["recall@5", "MRR", "precision@5"], ascending=False).iloc[0]["strategy"]
print("Выбранная стратегия чанкинга:", best_strategy)
print(summary.to_string(index=False))


Выбранная стратегия чанкинга: small_500_100
      strategy  precision@5  recall@5      MRR  recall@1  recall@3  recall@10  recall@20
medium_900_150     0.146429  0.562500 0.417792  0.169643       0.5   0.669643   0.785714
 small_500_100     0.153571  0.589286 0.465340  0.258929       0.5   0.687500   0.776786


### Вывод по шагу 3

После локального запуска нужно сравнить агрегированные метрики `summary` и `by_kind`, выбрать стратегию по `recall@5`, `MRR`, `precision@5` и зафиксировать причины выбора. Выбранная стратегия используется в шаге 4 для baseline retrieval и диагностики ошибок. Финальный контрольный прогон выполняется на удалённой машине только после успешного локального запуска.
